In [11]:
import warnings

# Suppress all UserWarning messages
warnings.filterwarnings("ignore", category=UserWarning)

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, TensorBoard
import pickle
import datetime

In [ ]:
data = pd.read_csv("Churn_Modelling.csv")
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
# Encoding the gender using Label Encoder
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

# One-Hot Encoding Geography
onehot_encode_geo = OneHotEncoder()
geo_encoded = onehot_encode_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns = onehot_encode_geo.get_feature_names_out(['Geography']))

# Concatinating the data
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

X = data.drop('Exited',axis=1)
y = data['Exited']

In [4]:
# Performing Train Test Split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Scaling the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [5]:
# Saving the encoded sets and scaled sets
with open('label_encoder_gender_hpt.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encode_geo_hpt.pkl','wb') as file:
    pickle.dump(onehot_encode_geo,file)

with open('scaler_hpt.pkl','wb') as file:
    pickle.dump(scaler,file)

In [6]:
# Define a function to create a model and try different models --> use keras classifier

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='adam',loss = 'binary_crossentropy', metrics=['accuracy'])

    return model


In [7]:
# Create a Keras Classifier

model = KerasClassifier(layers=1,neurons=32,model=create_model,epochs=50, batch_size=10, verbose=0)


In [8]:
# Defining the Grid Search parameters
param_grid = {
    'neurons': [16,32,64,128],
    'layers' : [1,2],
    #'batch_size' : [10,20],
    'epochs' : [50,100]
}


In [9]:
# Performing Grid Search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=2)

print("Starting Exhaustive Grid Search on CPU...")
grid_result = grid.fit(X_train,y_train)

print("\n=== GRID SEARCH COMPLETE ===")
# Printing the best parameters
#print("Best : %f using %s" % (grid_result.best_score_, grid_result.best_params_))
print("Best Score: %f" % grid_result.best_score_)
print("Best Parameter Combination Found:")
for param, value in grid_result.best_params_.items():
    print(f" -> {param}: {value}")

Starting Exhaustive Grid Search on CPU...
Fitting 3 folds for each of 16 candidates, totalling 48 fits

=== GRID SEARCH COMPLETE ===
Best Score: 0.856499
Best Parameter Combination Found:
 -> epochs: 100
 -> layers: 1
 -> neurons: 16


In [24]:
# Setting up log directory
log_dir = 'hptlogs/grid_search/' +datetime.datetime.now().strftime('%y%m%d-%H%M%S')
tb_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [28]:
# Re-creating optimal model structure discovered by grid search
# best params : layers=1, neurons=16, epochs=100
optimal_model = create_model(neurons=16,layers=1)

print("Training the final optimal model and saving tensor board logs..")
# Fitting the model on full training set withcallback enabled
optimal_model.fit(
    X_train,
    y_train,
    validation_data=(X_test,y_test),
    epochs=100,
    batch_size=10,
    callbacks = [tb_callback],
    verbose=1
)

# Saving the model
optimal_model.save('optimal_model.h5')
print("\n Training complete. Model saved as 'optimal_model.h5'")

Training the final optimal model and saving tensor board logs..
Epoch 1/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.7732 - loss: 0.4926 - val_accuracy: 0.8205 - val_loss: 0.4238
Epoch 2/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8169 - loss: 0.4249 - val_accuracy: 0.8335 - val_loss: 0.4010
Epoch 3/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8316 - loss: 0.4018 - val_accuracy: 0.8430 - val_loss: 0.3793
Epoch 4/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8411 - loss: 0.3808 - val_accuracy: 0.8520 - val_loss: 0.3609
Epoch 5/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8490 - loss: 0.3655 - val_accuracy: 0.8610 - val_loss: 0.3523
Epoch 6/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8520 - loss: 0.3568 - val_accuracy: 0.8600 - val_loss: 0.3478
Epoch 7/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8543 - loss: 0.3515 - val_accuracy: 0.8615 - val_loss: 0.3465
Epoch 8/100
800/800 ━━━━━━━━━━━━━


 Training complete. Model saved as 'optimal_model.h5'


In [29]:
%load_ext tensorboard
%tensorboard --logdir hptlogs/grid_search/

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Launching TensorBoard...